<a href="https://colab.research.google.com/github/tevfikaytekin/recommender_systems_course/blob/main/baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recommender Systems: Establishing Baselines

Before diving into complex machine learning models for recommendation systems, it is crucial to establish robust **baselines**. A baseline is a simple, heuristic-based approach that provides a minimum performance threshold. Any advanced algorithm (like Collaborative Filtering, Matrix Factorization, or Deep Learning models) must demonstrate significantly better performance than these baselines to justify their added computational complexity.

In this notebook, we will explore simple baseline methods using the popular **MovieLens (ml-latest-small)** dataset. We will cover two primary evaluation tasks in recommender systems:

1. **Rating Prediction:** Predicting the exact rating (e.g., 1 to 5 stars) a user would give to an unseen movie. We will evaluate random guessing, global averages, and user/item-specific averages using MAE and RMSE metrics.
2. **Top-N Recommendation:** Suggesting a ranked list of $N$ movies a user is most likely to interact with or enjoy. We will evaluate random ranking, highest average rating, and the universally strong **Popularity** baseline using the Hit Ratio (HR) metric.

Let's get started by setting up our environment and loading the data!

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from scipy.sparse import csr_matrix
import copy

### Movielens ml-latest-small dataset

In [1]:
!wget --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -o ml-latest-small.zip

--2026-09-22 12:09:03--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K  --.-KB/s    in 0.1s    

2026-09-22 12:09:04 (7.28 MB/s) - ‘ml-latest-small.zip’ saved [978202/978202]

Archive:  ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


In [2]:
with open('ml-latest-small/README.txt', 'r') as f:
    print(f.read())

Summary

This dataset (ml-latest-small) describes 5-star rating and free-text tagging activity from [MovieLens](http://movielens.org), a movie recommendation service. It contains 100836 ratings and 3683 tag applications across 9742 movies. These data were created by 610 users between March 29, 1996 and September 24, 2018. This dataset was generated on September 26, 2018.

Users were selected at random for inclusion. All selected users had rated at least 20 movies. No demographic information is included. Each user is represented by an id, and no other information is provided.

The data are contained in the files `links.csv`, `movies.csv`, `ratings.csv` and `tags.csv`. More details about the contents and use of all these files follows.

This is a *development* dataset. As such, it may change over time and is not an appropriate dataset for shared research results. See available *benchmark* datasets if that is your intent.

This and other GroupLens data sets are publicly available for down

In [5]:
ratings = pd.read_csv("ml-latest-small/ratings.csv", sep=",")
print(ratings.shape)
ratings.head(10)

(100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
5,1,70,3.0,964982400
6,1,101,5.0,964980868
7,1,110,4.0,964982176
8,1,151,5.0,964984041
9,1,157,5.0,964984100


In [6]:
n_users = ratings.iloc[:,0].unique().size
n_items = ratings.iloc[:,1].unique().size
n_ratings = ratings.iloc[:,1].size
users = ratings.iloc[:,0].unique()
items = ratings.iloc[:,1].unique()

print("Number of users:",n_users)
print("Number of items:",n_items)
print("Number of ratings:",n_ratings)
print("Sparsity:",n_ratings/(n_users*n_items))

Number of users: 610
Number of items: 9724
Number of ratings: 100836
Sparsity: 0.016999683055613623


In [7]:
links = pd.read_csv("ml-latest-small/links.csv", sep=",")
print(links.shape)
links.head()

(9742, 3)


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [8]:
movies = pd.read_csv("ml-latest-small/movies.csv", sep=",")
print(movies.shape)
movies.head()

(9742, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [9]:
tags = pd.read_csv("ml-latest-small/tags.csv", sep=",")
print(tags.shape)
tags.head()

(3683, 4)


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


# Rating Prediction

### Random Prediction

Randomly makes a rating prediction in the range 1 to 5.

In [10]:
error = 0
for i in range(n_ratings):
    error += np.abs(ratings.iloc[i,2] - np.random.randint(1,6))
print("Mean Absolute Error (MAE): ",error/n_ratings)

Mean Absolute Error (MAE):  1.4836566305684478


In [11]:
error = 0
for i in range(n_ratings):
    error += (ratings.iloc[i,2] - np.random.randint(1,6))**2
print("Root Means Squared Error (RMSE): ",np.sqrt(error/n_ratings))

Root Means Squared Error (RMSE):  1.8267722545970029


### Average Prediction

Always makes the same prediction which is the average rating of all users. This can be improved as follows: For a given $(u,i)$ pair, make the prediction as the average rating of user $u$ or the average rating of item $i$. As can be tested below, it is much better than random prediction.

In [12]:
X_train, X_test = train_test_split(ratings, test_size=0.1)
train_size = X_train.shape[0]
test_size = X_test.shape[0]

avg_rating = X_train.iloc[:,2].mean()
print("Avg. rating:",avg_rating)
error = 0
for i in range(test_size):
    error += np.abs(X_test.iloc[i,2] - avg_rating)
print(error/test_size)

Avg. rating: 3.4984022390691116
0.8159597581129677


In [13]:
item_averages = X_train.groupby('movieId')['rating'].mean()
user_averages = X_train.groupby('userId')['rating'].mean()

print("Item Averages (first 5):\n", item_averages.head())
print("\nUser Averages (first 5):\n", user_averages.head())

Item Averages (first 5):
 movieId
1    3.907895
2    3.388298
3    3.295918
4    2.250000
5    3.047619
Name: rating, dtype: float64

User Averages (first 5):
 userId
1    4.355140
2    3.910714
3    2.470588
4    3.520000
5    3.717949
Name: rating, dtype: float64


Let's evaluate the predictions using **Item Averages**. If an item is not present in the training set, we will default to the global average.

In [15]:
# Prediction using Item Averages
item_avg_preds = []
error_item_avg = 0
for index, row in X_test.iterrows():
    user_id, movie_id, true_rating = row['userId'], row['movieId'], row['rating']

    # Get item average, default to global average if item not in training
    pred_rating = item_averages.get(movie_id, avg_rating)

    item_avg_preds.append((user_id, movie_id, true_rating, pred_rating))
    error_item_avg += np.abs(true_rating - pred_rating)

mae_item_avg = error_item_avg / test_size
print(f"Mean Absolute Error (MAE) using Item Averages: {mae_item_avg:.4f}")

error_item_avg_rmse = 0
for pred in item_avg_preds:
    error_item_avg_rmse += (pred[2] - pred[3])**2
rmse_item_avg = np.sqrt(error_item_avg_rmse / test_size)
print(f"Root Mean Squared Error (RMSE) using Item Averages: {rmse_item_avg:.4f}")

Mean Absolute Error (MAE) using Item Averages: 0.7420
Root Mean Squared Error (RMSE) using Item Averages: 0.9582


Now, let's evaluate the predictions using **User Averages**. Similarly, if a user is not present in the training set, we will default to the global average.

In [16]:
# Prediction using User Averages
user_avg_preds = []
error_user_avg = 0
for index, row in X_test.iterrows():
    user_id, movie_id, true_rating = row['userId'], row['movieId'], row['rating']

    # Get user average, default to global average if user not in training
    pred_rating = user_averages.get(user_id, avg_rating)

    user_avg_preds.append((user_id, movie_id, true_rating, pred_rating))
    error_user_avg += np.abs(true_rating - pred_rating)

mae_user_avg = error_user_avg / test_size
print(f"Mean Absolute Error (MAE) using User Averages: {mae_user_avg:.4f}")

error_user_avg_rmse = 0
for pred in user_avg_preds:
    error_user_avg_rmse += (pred[2] - pred[3])**2
rmse_user_avg = np.sqrt(error_user_avg_rmse / test_size)
print(f"Root Mean Squared Error (RMSE) using User Averages: {rmse_user_avg:.4f}")

Mean Absolute Error (MAE) using User Averages: 0.7218
Root Mean Squared Error (RMSE) using User Averages: 0.9262


## Top-N Recommendation

In this section, we measure the Top-N recommendation performance using the **Hit Ratio** metric. For each user, we sample one item they rated 5 stars and mix it with 99 randomly chosen items they haven't rated. We then rank these 100 items using a baseline approach (Popularity) and check if the 5-star item appears in the top $k$ recommendations.

In [20]:
X_train.groupby('movieId')['rating'].count()

,rating
movieId,
1,190
2,94
3,49
4,6
5,42
...,...
193581,1
193583,1
193585,1


In [22]:
import random

# Get popularity counts for each item from the training set
item_popularity = X_train.groupby('movieId')['rating'].count().to_dict()
all_items = set(ratings['movieId'].unique())

# Create a dictionary of items rated by each user to easily find unrated items
user_rated_items = ratings.groupby('userId')['movieId'].apply(set).to_dict()

hits_at_5 = 0
hits_at_10 = 0
hits_at_20 = 0
users_evaluated = 0

# Set random seed for reproducibility
random.seed(42)

for u in users:
    # Find items rated 5 by this user
    user_5_star_items = ratings[(ratings['userId'] == u) & (ratings['rating'] == 5)]['movieId'].tolist()

    if not user_5_star_items:
        continue # Skip users who haven't rated any item 5

    # Take exactly ONE item rated 5
    positive_item = random.choice(user_5_star_items)

    # Find 99 items NOT rated by this user
    unrated_items = list(all_items - user_rated_items[u])
    if len(unrated_items) < 99:
        continue

    negative_items = random.sample(unrated_items, 99)

    # Combine the 1 positive and 99 negative items
    test_items_u = [positive_item] + negative_items

    # Score them using the Popularity baseline
    scores = [(item, item_popularity.get(item, 0)) for item in test_items_u]

    # Rank items based on score (descending)
    scores.sort(key=lambda x: x[1], reverse=True)

    # Extract top-K items
    top_5 = [x[0] for x in scores[:5]]
    top_10 = [x[0] for x in scores[:10]]
    top_20 = [x[0] for x in scores[:20]]

    # Check for Hit
    if positive_item in top_5:
        hits_at_5 += 1
    if positive_item in top_10:
        hits_at_10 += 1
    if positive_item in top_20:
        hits_at_20 += 1

    users_evaluated += 1

print(f"Evaluated {users_evaluated} users.")
print(f"Popular Baseline Hit Ratio @ 5: {hits_at_5 / users_evaluated:.4f}")
print(f"Popular Baseline Hit Ratio @ 10: {hits_at_10 / users_evaluated:.4f}")
print(f"Popular Baseline Hit Ratio @ 20: {hits_at_20 / users_evaluated:.4f}")

Evaluated 573 users.
Popular Baseline Hit Ratio @ 5: 0.6841
Popular Baseline Hit Ratio @ 10: 0.8133
Popular Baseline Hit Ratio @ 20: 0.9180


Now, let's see how a completely **Random Baseline** performs in this Top-N recommendation task. Instead of scoring items by popularity, we simply rank the 100 items (1 positive + 99 negative) randomly.

In [25]:
hits_at_5_random = 0
hits_at_10_random = 0
hits_at_20_random = 0
users_evaluated_random = 0

# Set random seed for reproducibility
random.seed(42)

for u in users:
    # Find items rated 5 by this user
    user_5_star_items = ratings[(ratings['userId'] == u) & (ratings['rating'] == 5)]['movieId'].tolist()

    if not user_5_star_items:
        continue # Skip users who haven't rated any item 5

    # Take exactly ONE item rated 5
    positive_item = random.choice(user_5_star_items)

    # Find 99 items NOT rated by this user
    unrated_items = list(all_items - user_rated_items[u])
    if len(unrated_items) < 99:
        continue

    negative_items = random.sample(unrated_items, 99)

    # Combine the 1 positive and 99 negative items
    test_items_u = [positive_item] + negative_items

    # Randomly shuffle the items to simulate a random recommendation
    random.shuffle(test_items_u)

    # Extract top-K items
    top_5 = test_items_u[:5]
    top_10 = test_items_u[:10]
    top_20 = test_items_u[:20]

    # Check for Hit
    if positive_item in top_5:
        hits_at_5_random += 1
    if positive_item in top_10:
        hits_at_10_random += 1
    if positive_item in top_20:
        hits_at_20_random += 1

    users_evaluated_random += 1

print(f"Evaluated {users_evaluated_random} users.")
print(f"Random Baseline Hit Ratio @ 5: {hits_at_5_random / users_evaluated_random:.4f}")
print(f"Random Baseline Hit Ratio @ 10: {hits_at_10_random / users_evaluated_random:.4f}")
print(f"Random Baseline Hit Ratio @ 20: {hits_at_20_random / users_evaluated_random:.4f}")

Evaluated 573 users.
Random Baseline Hit Ratio @ 5: 0.0471
Random Baseline Hit Ratio @ 10: 0.1030
Random Baseline Hit Ratio @ 20: 0.1902


### Highest Average Rating Baseline

Similar to how we used item averages for rating prediction, we can use the **Average Rating** of an item to rank them for Top-N recommendation. We will rank the 100 test items based on their average rating in the training set (defaulting to the global average if unseen).

In [ ]:
# Get average rating for each item from the training set
item_avg_rating = X_train.groupby('movieId')['rating'].mean().to_dict()

hits_at_5_avg = 0
hits_at_10_avg = 0
hits_at_20_avg = 0
users_evaluated_avg = 0

# Set random seed for reproducibility
random.seed(42)

for u in users:
    # Find items rated 5 by this user
    user_5_star_items = ratings[(ratings['userId'] == u) & (ratings['rating'] == 5)]['movieId'].tolist()

    if not user_5_star_items:
        continue # Skip users who haven't rated any item 5

    # Take exactly ONE item rated 5
    positive_item = random.choice(user_5_star_items)

    # Find 99 items NOT rated by this user
    unrated_items = list(all_items - user_rated_items[u])
    if len(unrated_items) < 99:
        continue

    negative_items = random.sample(unrated_items, 99)

    # Combine the 1 positive and 99 negative items
    test_items_u = [positive_item] + negative_items

    # Score them using the Average Rating baseline
    # We use avg_rating (global average) for items not in the training set
    scores = [(item, item_avg_rating.get(item, avg_rating)) for item in test_items_u]

    # Rank items based on score (descending)
    # If there's a tie (e.g. many items with 5.0), we also shuffle to avoid bias
    random.shuffle(scores)
    scores.sort(key=lambda x: x[1], reverse=True)

    # Extract top-K items
    top_5 = [x[0] for x in scores[:5]]
    top_10 = [x[0] for x in scores[:10]]
    top_20 = [x[0] for x in scores[:20]]

    # Check for Hit
    if positive_item in top_5:
        hits_at_5_avg += 1
    if positive_item in top_10:
        hits_at_10_avg += 1
    if positive_item in top_20:
        hits_at_20_avg += 1

    users_evaluated_avg += 1

print(f"Evaluated {users_evaluated_avg} users.")
print(f"Highest Average Rating Baseline Hit Ratio @ 5: {hits_at_5_avg / users_evaluated_avg:.4f}")
print(f"Highest Average Rating Baseline Hit Ratio @ 10: {hits_at_10_avg / users_evaluated_avg:.4f}")
print(f"Highest Average Rating Baseline Hit Ratio @ 20: {hits_at_20_avg / users_evaluated_avg:.4f}")

Evaluated 573 users.
Highest Average Rating Baseline Hit Ratio @ 5: 0.0419
Highest Average Rating Baseline Hit Ratio @ 10: 0.2251
Highest Average Rating Baseline Hit Ratio @ 20: 0.4380


### Interpretation of Top-N Recommendation Baselines

From the results of our three baselines, we can observe the following:

1. **Popularity Baseline (Strongest):** Recommending the most popular items yields by far the highest Hit Ratios (e.g., HR@20 > 0.91). This is a common phenomenon in recommendation systems; popular items are popular for a reason, and most users have interacted with them. It serves as a very tough baseline to beat for personalized models.
2. **Highest Average Rating Baseline (Weak):** While better than random guessing, ranking purely by average rating performs poorly compared to popularity. This happens because "niche" items with only 1 or 2 perfect 5-star ratings get pushed to the very top, burying the items that are genuinely liked by a large, broad audience.
3. **Random Baseline (Lower Bound):** As expected, randomly ranking items performs the worst and provides an absolute lower bound to verify that our other methods are actually effective.

**Conclusion:** A sophisticated recommendation algorithm (like Collaborative Filtering or Matrix Factorization) should aim to approach or exceed the Hit Ratio of the Popularity baseline, while simultaneously providing much more *personalized* and *novel* recommendations, rather than just suggesting the exact same popular blockbusters to every single user.